# Create `type1_dev_artifacts.zip`

Notebook chuyên dụng để tạo artifact cho Claude/dev branch.

Nó sẽ:
1. Tự ghi `engine_v40_snapshot.py`.
2. Tự tìm hoặc tự tạo `*_flat_normalized.jsonl`.
3. Tự tìm hoặc tự tạo `type1_aug_dev_minimal.jsonl` và `type1_aug_holdout_minimal.jsonl`.
4. Gom probe outputs nếu có.
5. Tạo `/kaggle/working/type1_dev_artifacts.zip`.

Zip này mở khóa cả:
- YNU holdout spurious-fire check.
- Atom representation v2 development/gate.


In [ ]:
from pathlib import Path
import os, glob, json, re, subprocess, sys, shutil, zipfile, hashlib, datetime

WORK = Path("/kaggle/working") if Path("/kaggle").exists() else Path(".")
WORK.mkdir(parents=True, exist_ok=True)

# Optional overrides
CREATE_FLAT_IF_MISSING = True
CREATE_AUG_IF_MISSING = True

ENGINE_PATH = WORK / "engine_v40_snapshot.py"
NORMALIZER = WORK / "normalize_logic_dataset_to_flat.py"
AUGMENTER = WORK / "create_minimal_type1_aug_holdout.py"
STAGE = WORK / "type1_dev_artifacts"
ZIP_PATH = WORK / "type1_dev_artifacts.zip"

print("WORK =", WORK)


In [ ]:
# Write embedded engine + helper scripts
ENGINE_PATH.write_text('# --- v40 entity-conjunctive engine (frozen baseline; symbolic prehandler) ---\nimport re\n# -*- coding: utf-8 -*-\n"""v40.3 entity-grounded conjunctive Horn engine for the REAL Phase-1 distribution.\nPropositional over a single entity: facts = literals, rules = (conj of literals)->literal.\nForward-chain; answer options by derivability. Certificate = premise indices. Abstain-safe."""\nimport re\nSTOP={\'a\',\'an\',\'the\',\'of\',\'to\',\'in\',\'on\',\'at\',\'for\',\'and\',\'or\',\'that\',\'this\',\'their\',\'its\',\'it\',\'they\',\'them\',\n      \'is\',\'are\',\'was\',\'were\',\'be\',\'been\',\'has\',\'have\',\'had\',\'then\',\'if\',\'no\',\'not\',\'with\',\'as\',\'by\',\'from\',\n      \'artifact\',\'package\',\'manuscript\',\'sample\',\'batch\',\'item\',\'device\',\'record\',\'file\',\'student\'}\ndef _stem(t):\n    if re.search(r\'(ss|us|is)$\',t): return t\n    if re.search(r\'(ches|shes|xes|zes|ses)$\',t): return t[:-2]\n    if re.search(r\'ies$\',t): return t[:-3]+\'y\'\n    if t.endswith(\'s\'): t=t[:-1]\n    return re.sub(r\'(ing|ed)$\',\'\',t)\ndef atom_key(phrase):\n    s=re.sub(r\'(?<!^)(?=[A-Z])\',\' \',str(phrase)).lower()\n    nums=re.findall(r\'\\d+\', s)\n    toks=[ _stem(w) for w in re.findall(r\'[a-zA-Z]+\', s) ]\n    toks=[t for t in toks if t and t not in STOP and len(t)>2]\n    keys=sorted(set(toks))+["N"+n for n in sorted(set(nums))]   # keep numeric literals\n    return "".join(w.capitalize() for w in keys) if keys else None\n\n# split a clause into (atom, neg). Handles "X has Y", "X is Y", "X has no Y", "X lacks Y", "cannot ...", "is not ..."\n_LEAD=re.compile(r"^\\s*(if|then|that|who|which|it|its|their|this)\\b",re.I)\n_VERB=re.compile(r"\\b(cannot|can not|can|could|may|might|must|should|shall|will|would|is not|are not|was not|were not|isn\'t|aren\'t|is|are|was|were|has no|have no|had no|has|have|had|lacks?|without|requires?|needs?|contains?|completed?|enters?|gains?|receives?|provides?|shows?|states?|holds?|carries|monitors?|captures?|eligible|allowed|approved|assigned|be|been|being)\\b",re.I)\nACTION_VERBS={\'receives\',\'receive\',\'provides\',\'provide\',\'shows\',\'show\',\'states\',\'state\',\'monitors\',\'monitor\',\'captures\',\'capture\',\'enters\',\'enter\',\'requires\',\'require\',\'needs\',\'need\',\'gains\',\'gain\',\'completed\',\'complete\',\'contains\',\'contain\',\'reports\',\'report\',\'releases\',\'release\',\'passes\',\'pass\',\'improves\',\'improve\',\'supports\',\'support\',\'recommends\',\'recommend\',\'administered\',\'administer\',\'approved\',\'approve\'}\n_NEGWORD=re.compile(r\'\\b(no|not|cannot|can not|lacks?|without|isn\\\'t|aren\\\'t|never|nor|incomplete|missing|lacking)\\b\',re.I)\ndef to_literal(clause):\n    c=clause.strip().rstrip(\'.\').strip()\n    c=_LEAD.sub(\'\',c).strip()\n    neg=bool(re.search(r"\\b(no|not|cannot|can not|never|lacks?|without|isn\'t|aren\'t|incomplete|missing|lacking|nor|un(?:able|verified|established))\\b",c,re.I))\n    m=_VERB.search(c)\n    pred=c[m.end():].strip() if m else c\n    verb=(m.group(1).lower() if m else \'\')\n    if m and verb in ACTION_VERBS:\n        pred = verb + \' \' + pred\n    # peel any leftover leading modal/aux/passive markers and articles\n    for _ in range(4):\n        pred=re.sub(r"^\\s*(be|been|being|to|a|an|the|no|not|its|their)\\b","",pred,flags=re.I).strip()\n    a=atom_key(pred)\n    return (a,neg) if a else None\n\ndef parse_premise(p):\n    s=str(p).strip()\n    m=re.search(r\'^\\s*if\\b(.+?),?\\s*\\bthen\\b(.+)$\',s,re.I)\n    if m:\n        ante=re.split(r\'\\band\\b\',m.group(1),flags=re.I)\n        lits=[to_literal(x) for x in ante]; lits=[l for l in lits if l]\n        con=to_literal(m.group(2))\n        if con and lits: return (\'rule\',lits,con)\n        return None\n    # Universal relative rule: Every/All <role> <condition> <verb> <consequent>\n    # Examples: Every volunteer assigned to shift receives badge; All satellites with cameras can capture images.\n    m2=re.search(r\'^\\s*(every|all)\\s+[a-zA-Z]+s?\\s+(.+?)\\s+\\b(can|may|must|should|will|would|receives?|gets?|gains?|provides?|captures?|monitors?|requires?|needs?|is|are)\\b\\s+(.+)$\',s,re.I)\n    if m2:\n        cond=m2.group(2).strip()\n        cons=(m2.group(3)+" "+m2.group(4)).strip()\n        litc=to_literal(cond); litd=to_literal(cons)\n        if litc and litd: return (\'rule\',[litc],litd)\n    if re.search(r\'^\\s*(no premise|it (is|cannot)|unknown|there is no information)\',s,re.I): return None\n    lit=to_literal(s)\n    return (\'fact\',lit) if lit else None\n\ndef solve_entity(premises):\n    facts={}  # atom -> (bool_value, premise_idx)\n    rules=[]\n    for i,p in enumerate(premises):\n        pp=parse_premise(p)\n        if not pp: continue\n        if pp[0]==\'fact\':\n            a,neg=pp[1]; facts.setdefault(a,(not neg,[i]))\n        else:\n            rules.append((i,pp[1],pp[2]))\n    changed=True\n    while changed:\n        changed=False\n        for i,lits,con in rules:\n            ca,cneg=con\n            ok=True; path=[i]\n            for a,neg in lits:\n                if a in facts and facts[a][0]==(not neg): path+=facts[a][1]\n                else: ok=False; break\n            if ok and ca not in facts:\n                facts[ca]=((not cneg),sorted(set(path))); changed=True\n    return facts\n\n_META_RE=__import__("re").compile(r"\\b(not (?:yet )?(?:established|confirmed|verified|approved|cleared|determined)|cannot be (?:established|confirmed)|unsupported|is not established|no premise|undetermined|not (?:available|present))\\b",__import__("re").I)\ndef decompose_option(opt):\n    import re\n    t=re.sub(r\'^\\s*[A-Da-d][.):]\\s*\',\'\',str(opt)).strip()\n    t=re.split(r\'\\bbecause\\b\', t, maxsplit=1, flags=re.I)[0].strip()  # drop causal justification\n    parts=re.split(r\',\\s*but\\s+|\\s+but\\s+|;\\s+|\\s+while\\s+|\\s+whereas\\s+|\\s+and\\s+\',t,flags=re.I)\n    claims=[]\n    for p in parts:\n        p=p.strip()\n        if not p: continue\n        is_meta=bool(_META_RE.search(p))\n        lit=to_literal(p)\n        claims.append((lit,is_meta,p))\n    return claims\n\ndef answer_mc(premises, options):\n    facts=solve_entity(premises)\n    res={}\n    for lab,opt in zip("ABCD",options):\n        claims=decompose_option(opt)\n        if not claims: res[lab]=(\'UNSUP\',[]); continue\n        status=\'PROVEN\'; path=[]\n        for lit,is_meta,txt in claims:\n            if lit is None: status=\'UNSUP\'; break\n            a,neg=lit; have = a in facts\n            val = facts[a][0] if have else None\n            if is_meta:\n                # meta "not established": correct only if NOT positively proven\n                if have and val==True: status=\'DISPROVEN\'; break\n            else:\n                if have and val==(not neg): path+=facts[a][1]\n                elif have and val==(neg): status=\'DISPROVEN\'; break\n                else: status=\'UNSUP\'; break\n        res[lab]=(status, sorted(set(path)))\n    proven=[l for l in res if res[l][0]==\'PROVEN\']\n    if len(proven)==1: return proven[0],res[proven[0]][1],\'entity_unique_proof\',res\n    return None,[],(\'multiple\' if proven else \'none\'),res\n\n\n# ================= Phase-1 REPLAY HARNESS =================\ndef _opt_texts(rp):\n    import re\n    f=[o[1].strip().replace("\\n"," ") for o in re.findall(r"(?:^|\\n)\\s*([A-D])[.)]\\s*(.+?)(?=\\n\\s*[A-D][.)]|\\Z)",rp.get("query",""),flags=re.S)]\n    return f if len(f)>=2 else (rp.get("options") or [])\ndef replay_phase1(path):\n    import json,re\n    d=json.load(open(path)); t1=[l for l in d["logs"] if l.get("type")=="type1"]\n    fired=aok=pok=0; fixed=[]; wrong=[]; abst=0\n    for l in t1:\n        rp=l["request_payload"]; exp=l.get("expected") or {}; ea=str(exp.get("answer","")).strip().upper()\n        opts=_opt_texts(rp)\n        if not opts: continue\n        a,pu,why,res=answer_mc(rp.get("premises",[]) or [], opts)\n        if a is None: abst+=1; continue\n        fired+=1; ok=(a==ea); aok+=ok; pok+=(sorted(pu)==sorted(exp.get("premises_used") or []))\n        if ok and l.get("status")!="correct": fixed.append(l["query_id"])\n        if not ok: wrong.append({"query_id":l["query_id"],"exp":ea,"got":a})\n    rep={"n":len(t1),"fired":fired,"answer_correct":aok,"premises_used_correct":pok,\n         "wrong":wrong,"fixed_old_wrong":fixed,"abstained":abst,\n         "precision_when_fired":round(aok/max(fired,1),3),"coverage":round(fired/max(len(t1),1),3),\n         "gate":"ABSTAIN_SAFE" if not wrong else "HAS_WRONG_FIX_BEFORE_APPLY"}\n    return rep\ndef _autofind():\n    import glob,os,sys\n    if len(sys.argv)>1 and os.path.exists(sys.argv[1]): return sys.argv[1]\n    for c in ["exact_eval_round1_Astatine.json","/kaggle/input/**/exact_eval_round1_Astatine.json",\n              "/kaggle/working/exact_eval_round1_Astatine.json","./exact_eval_round1_Astatine.json"]:\n        h=sorted(glob.glob(c,recursive=True)) if any(x in c for x in "*?[") else ([c] if os.path.exists(c) else [])\n        if h: return h[0]\n    return None\nprint(\'v40 symbolic engine loaded\')', encoding="utf-8")
NORMALIZER.write_text('\n#!/usr/bin/env python3\nimport json, re, argparse, pathlib\n\ndef extract_options(question):\n    matches = re.findall(r"(?:^|\\n)\\s*([A-D])[.)]\\s*(.+?)(?=\\n\\s*[A-D][.)]|\\Z)", str(question), flags=re.S)\n    return [m[0] for m in matches[:4]] if len(matches) >= 2 else []\n\ndef build_prompt(premises, question, answer_space):\n    hint = "<A, B, C, or D>" if answer_space == "mc" else "<Yes, No, or Unknown>"\n    lines = ["Reason step by step:", "Use only the given premises. Do not use outside knowledge.", "", "Premises:"]\n    lines += [f"{i}. {p}" for i, p in enumerate(premises, 1)]\n    lines += ["", "Question:", str(question), "", f"End with exactly one line: Final Answer: {hint}"]\n    return "\\n".join(lines)\n\ndef normalize_record_file(input_json, output_jsonl, dataset_name="dataset"):\n    data = json.load(open(input_json, encoding="utf-8"))\n    n_rows = 0\n    with open(output_jsonl, "w", encoding="utf-8") as f:\n        for i, rec in enumerate(data):\n            premises_nl = rec.get("premises-NL", rec.get("premises_NL", [])) or []\n            premises_fol = rec.get("premises-FOL", rec.get("premises_FOL", [])) or []\n            questions = rec.get("questions", [])\n            answers = rec.get("answers", [])\n            explanations = rec.get("explanation", rec.get("explanations", []))\n            idxs = rec.get("idx", [])\n            if not isinstance(questions, list): questions = [questions]\n            if not isinstance(answers, list): answers = [answers]\n            for j, q in enumerate(questions):\n                gold = answers[j] if j < len(answers) else None\n                gold = "Unknown" if str(gold).strip() == "Uncertain" else gold\n                options = extract_options(q)\n                is_mc = bool(options) or str(gold).strip() in list("ABCD")\n                is_ynu = str(gold).strip() in {"Yes", "No", "Unknown", "Uncertain"}\n                idx = idxs[j] if isinstance(idxs, list) and j < len(idxs) else []\n                row = {\n                    "flat_id": f"{dataset_name}:{i}:{j}",\n                    "source_dataset": dataset_name,\n                    "source_record_id": i,\n                    "question_id_in_record": j,\n                    "premises_NL": premises_nl,\n                    "premises_FOL": premises_fol,\n                    "question": q,\n                    "gold": gold,\n                    "options": options,\n                    "is_mc": is_mc,\n                    "is_ynu": is_ynu,\n                    "premises_used": [x-1 for x in idx if isinstance(x, int)] if isinstance(idx, list) else [],\n                    "premises_used_gold_1based": idx,\n                    "premises_used_gold_0based": [x-1 for x in idx if isinstance(x, int)] if isinstance(idx, list) else [],\n                    "explanation": explanations[j] if isinstance(explanations, list) and j < len(explanations) else "",\n                    "prompt": build_prompt(premises_nl, q, "mc" if is_mc else "ynu"),\n                }\n                f.write(json.dumps(row, ensure_ascii=False) + "\\n")\n                n_rows += 1\n    return n_rows\n\ndef main():\n    ap = argparse.ArgumentParser()\n    ap.add_argument("input_json")\n    ap.add_argument("output_jsonl")\n    ap.add_argument("--dataset-name", default="dataset")\n    args = ap.parse_args()\n    n = normalize_record_file(args.input_json, args.output_jsonl, args.dataset_name)\n    print(json.dumps({"input": args.input_json, "output": args.output_jsonl, "rows": n}, indent=2))\n\nif __name__ == "__main__":\n    main()\n', encoding="utf-8")
AUGMENTER.write_text('\n#!/usr/bin/env python3\nimport json, argparse, pathlib\n\nRULE_SURFACES = [\n    "If an item has {a} and no {b}, then it is {c}.",\n    "When an item has {a} but no {b}, it is {c}.",\n    "Any item with {a} and without {b} is {c}.",\n    "An item that has {a} and lacks {b} is {c}.",\n]\nNAMES = ["Alpha", "Beta", "Gamma", "Delta", "Nova", "Vega", "Aster", "Orion"]\nA = ["humidity control", "medical priority", "ethics approval", "thermal sensor", "route clearance", "barcode scan"]\nB = ["pest report", "blocked route", "budget approval", "radar module", "renal screen", "manager signoff"]\nC = ["display ready", "aerial eligible", "study approved", "temperature monitored", "protocol entered", "review required"]\n\ndef sample(i, canary=False):\n    name = NAMES[i % len(NAMES)]\n    a = A[i % len(A)]\n    b = B[(i*3+1) % len(B)]\n    c = C[(i*5+2) % len(C)]\n    premises = [\n        RULE_SURFACES[i % len(RULE_SURFACES)].format(a=a, b=b, c=c),\n        f"Item {name} has {a}.",\n        f"Item {name} has no {b}.",\n    ]\n    options = [\n        f"Item {name} is {c}",\n        f"Item {name} has {b}",\n        f"Item {name} is not {c}",\n        f"Item {name} has unrelated clearance",\n    ]\n    gold = "A"\n    if canary:\n        d = "archive approved"\n        premises.append(f"If an item has {a}, then it is {d}.")\n        options[1] = f"Item {name} is {d}"\n        gold = "CANARY_MULTI_PROVEN"\n    question = (\n        "Based on the premises, which conclusion is logically supported?\\n"\n        f"A. {options[0]}\\nB. {options[1]}\\nC. {options[2]}\\nD. {options[3]}"\n    )\n    return {\n        "flat_id": f"aug_min:{i}",\n        "source_dataset": "aug_minimal_abstract",\n        "premises_NL": premises,\n        "premises_FOL": [],\n        "question": question,\n        "gold": gold,\n        "options": ["A", "B", "C", "D"],\n        "is_mc": True,\n        "is_ynu": False,\n        "premises_used": [0, 1, 2] if not canary else [],\n        "abstract_spec": {"a": a, "b": b, "c": c, "canary": canary},\n    }\n\ndef write_jsonl(path, rows):\n    with open(path, "w", encoding="utf-8") as f:\n        for r in rows:\n            f.write(json.dumps(r, ensure_ascii=False) + "\\n")\n\ndef main():\n    ap = argparse.ArgumentParser()\n    ap.add_argument("--outdir", default="/kaggle/working/aug_generated")\n    ap.add_argument("--dev", type=int, default=80)\n    ap.add_argument("--holdout", type=int, default=200)\n    args = ap.parse_args()\n    out = pathlib.Path(args.outdir); out.mkdir(parents=True, exist_ok=True)\n    dev = [sample(i, canary=(i % 5 == 0)) for i in range(args.dev)]\n    hold = [sample(1000+i, canary=(i % 4 == 0)) for i in range(args.holdout)]\n    write_jsonl(out / "type1_aug_dev_minimal.jsonl", dev)\n    write_jsonl(out / "type1_aug_holdout_minimal.jsonl", hold)\n    manifest = {\n        "note": "Minimal abstract-skeleton augmentation. Stress-test/canary only, not promotion evidence by itself.",\n        "dev_rows": len(dev),\n        "holdout_rows": len(hold),\n        "files": [str(out / "type1_aug_dev_minimal.jsonl"), str(out / "type1_aug_holdout_minimal.jsonl")]\n    }\n    (out / "manifest.json").write_text(json.dumps(manifest, indent=2, ensure_ascii=False), encoding="utf-8")\n    print(json.dumps(manifest, indent=2, ensure_ascii=False))\n\nif __name__ == "__main__":\n    main()\n', encoding="utf-8")

print("Wrote:", ENGINE_PATH, ENGINE_PATH.stat().st_size)
print("Wrote:", NORMALIZER, NORMALIZER.stat().st_size)
print("Wrote:", AUGMENTER, AUGMENTER.stat().st_size)


In [ ]:
# Auto-find helpers
SEARCH_ROOTS = [p for p in ["/kaggle/input", "/kaggle/working", ".", "/mnt/data"] if Path(p).exists()]

def find_files(patterns, max_hits=100):
    hits = []
    for root in SEARCH_ROOTS:
        for pat in patterns:
            hits += glob.glob(str(Path(root) / "**" / pat), recursive=True)
    seen, uniq = set(), []
    for h in hits:
        h = str(Path(h))
        if h not in seen and "__MACOSX" not in h:
            seen.add(h); uniq.append(h)
    def score(x):
        s = 0
        if "/kaggle/working/" in x: s -= 30
        if "/kaggle/input/" in x: s -= 20
        if "/mnt/data/" in x: s -= 10
        if x.endswith(".probe.json"): s += 50
        return (s, len(x), x)
    return sorted(uniq, key=score)[:max_hits]

def show(label, xs):
    print(f"\\n{label}: {len(xs)}")
    for x in xs[:30]:
        print(" -", x)
    if len(xs) > 30:
        print(" ...")


In [ ]:
# Find/create flat normalized files
flat_files = find_files([
    "*flat_normalized.jsonl",
    "v5_repair_flat_normalized.jsonl",
    "cleaned_v2_flat_normalized.jsonl",
], 100)
flat_files = [x for x in flat_files if not x.endswith(".probe.json")]
show("Existing flat normalized", flat_files)

created = []
if CREATE_FLAT_IF_MISSING:
    # We specifically want cleaned_v2 and v5_repair flats. If absent, create from record-level source.
    have_names = " ".join(Path(x).name for x in flat_files).lower()
    need_cleaned = "cleaned_v2" not in have_names
    need_v5 = "v5_repair" not in have_names and "tier1" not in have_names
    if need_cleaned or need_v5:
        record_sources = find_files([
            "Logic_Based_Educational_Queries_cleaned_v2*.json",
            "Logic_Based_Educational_Queries_v5_repair*.json",
        ], 100)
        show("Record-level sources", record_sources)
        for src in record_sources:
            name = Path(src).stem
            low = name.lower()
            if ("cleaned_v2" in low and need_cleaned) or ("v5_repair" in low and need_v5) or ("tier1" in low and need_v5):
                outp = WORK / f"{name}_flat_normalized.jsonl"
                cmd = [sys.executable, str(NORMALIZER), src, str(outp), "--dataset-name", name]
                r = subprocess.run(cmd, text=True, capture_output=True)
                print("\\nNORMALIZE:", src)
                print(r.stdout)
                if r.stderr:
                    print("STDERR:", r.stderr)
                if r.returncode == 0 and outp.exists():
                    created.append(str(outp))
    flat_files = find_files(["*flat_normalized.jsonl"], 100)
    flat_files = [x for x in flat_files if not x.endswith(".probe.json")]

show("Final flat normalized", flat_files)


In [ ]:
# Find/create augmentation files
aug_files = find_files([
    "type1_aug_dev*.jsonl",
    "type1_aug_holdout*.jsonl",
    "*aug*dev*.jsonl",
    "*aug*holdout*.jsonl",
], 100)
aug_files = [x for x in aug_files if not x.endswith(".probe.json")]
show("Existing aug files", aug_files)

if CREATE_AUG_IF_MISSING and not any("holdout" in Path(x).name.lower() for x in aug_files):
    outdir = WORK / "aug_generated"
    cmd = [sys.executable, str(AUGMENTER), "--outdir", str(outdir), "--dev", "80", "--holdout", "200"]
    r = subprocess.run(cmd, text=True, capture_output=True)
    print("\\nAUGMENT:")
    print(r.stdout)
    if r.stderr:
        print("STDERR:", r.stderr)
    if r.returncode != 0:
        raise RuntimeError("Augment generation failed")

aug_files = find_files([
    "type1_aug_dev*.jsonl",
    "type1_aug_holdout*.jsonl",
    "*aug*dev*.jsonl",
    "*aug*holdout*.jsonl",
], 100)
aug_files = [x for x in aug_files if not x.endswith(".probe.json")]
show("Final aug files", aug_files)


In [ ]:
# Gather optional probe outputs and phase1
probe_outputs = find_files(["*.probe.json"], 200)
phase1_files = find_files(["exact_eval_round1_Astatine.json"], 50)

show("Probe outputs", probe_outputs)
show("Phase1 files", phase1_files)


In [ ]:
# Create zip artifact
import shutil, json, hashlib, datetime
if STAGE.exists():
    shutil.rmtree(STAGE)
STAGE.mkdir(parents=True, exist_ok=True)

want = []
want += [str(ENGINE_PATH), str(NORMALIZER), str(AUGMENTER)]
want += flat_files
want += aug_files
want += probe_outputs
want += phase1_files[:1]  # include one Phase-1 if found

copied = []
seen = set()
for f in want:
    p = Path(f)
    if p.exists() and p.is_file() and str(p) not in seen:
        seen.add(str(p))
        dest = STAGE / p.name
        shutil.copy(p, dest)
        copied.append(dest)

def sha256(p):
    h = hashlib.sha256()
    with open(p, "rb") as f:
        for chunk in iter(lambda: f.read(1024*1024), b""):
            h.update(chunk)
    return h.hexdigest()

manifest = {
    "created_utc": datetime.datetime.utcnow().isoformat() + "Z",
    "purpose": "type1_dev_artifacts for YNU holdout spurious-check and atom representation v2 development",
    "required_for_claude": [
        "*cleaned_v2*_flat_normalized.jsonl",
        "*v5_repair*_flat_normalized.jsonl or *tier1*_flat_normalized.jsonl",
        "type1_aug_holdout_minimal.jsonl",
        "engine_v40_snapshot.py"
    ],
    "files": [
        {"name": p.name, "size": p.stat().st_size, "sha256": sha256(p)}
        for p in sorted(copied, key=lambda x: x.name)
    ],
}
(STAGE / "MANIFEST_type1_dev_artifacts.json").write_text(json.dumps(manifest, indent=2, ensure_ascii=False), encoding="utf-8")

if ZIP_PATH.exists():
    ZIP_PATH.unlink()
z = shutil.make_archive(str(ZIP_PATH).replace(".zip", ""), "zip", STAGE)

print("ZIP:", z)
print("STAGE:", STAGE)
print("FILES:")
for x in manifest["files"]:
    print(" -", x["name"], x["size"])
print("\\nMANIFEST:")
print(json.dumps(manifest, indent=2, ensure_ascii=False)[:5000])


In [ ]:
# Validate required contents
names = [p.name for p in STAGE.iterdir() if p.is_file()]
lower = " ".join(n.lower() for n in names)

checks = {
    "engine_v40_snapshot.py": "engine_v40_snapshot.py" in names,
    "cleaned_v2_flat": "cleaned_v2" in lower and "flat_normalized" in lower,
    "v5_repair_or_tier1_flat": (("v5_repair" in lower) or ("tier1" in lower)) and "flat_normalized" in lower,
    "aug_holdout": "holdout" in lower and "aug" in lower,
}
print(json.dumps(checks, indent=2))
if not all(checks.values()):
    print("\\nWARNING: zip created, but some recommended files are missing.")
    print("Attach/source missing raw record-level datasets if flat files could not be created.")
else:
    print("\\nOK: zip contains required core artifacts.")
print("Download:", ZIP_PATH)
